<a href="https://colab.research.google.com/github/0220nakarina-ui/welfare-scheduling-engine/blob/main/%E4%BB%8B%E8%AD%B7%E9%96%A2%E9%80%A3/%E4%BB%8B%E8%AD%B7%E7%8F%BE%E5%A0%B4%E3%82%B7%E3%83%95%E3%83%88%E4%BD%9C%E6%88%90%E7%B7%B4%E7%BF%92.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 職員データ作成(名前・資格・勤務時間・勤務形態)
from dataclasses import dataclass

@dataclass
class Staff:
  name: str              #名前
  license: str           #保有資格
  work_time: str         #勤務時間
  shift: list            #勤務形態(常勤・非常勤)

  def work_days(self):
          return self.shift.count("出勤")

# 職員データ
staffs = [
    Staff("田中","介護福祉士","常勤",[]),
    Staff("山田","介護福祉士","常勤",[]),
    Staff("佐藤","実務者研修","常勤",[]),
    Staff("山本","実務者研修","常勤",[]),
    Staff("如月","看護師","常勤",[]),
    Staff("酒井","看護師","常勤",[]),
    Staff("中川","看護師","常勤",[]),
    Staff("田栗","看護師","常勤",[]),
    Staff("藤原","初任者研修","非常勤",[]),
    Staff("青木","初任者研修","非常勤",[]),
    ]


weekdays = ["月", "火", "水", "木", "金", "土", "日"]

# 希望休
request_off = {
    "田中":[3,6],
    "酒井":[8],
    "如月":[9,25],
    "山田":[8,13],
}

# 初期化
def _init(staffs,days):
    for s in staffs:
      s.shift = ["公休"] * days

# 連勤チェック(最高５連勤まで)
def streak(staff,day):
     count = 0
     for i in range(day-1, -1, -1):
        if staff.shift[i] == "出勤":
          count += 1
        else:
            break
     return count

# 出勤可否
def can_work(staff,day):
  # 希望休は絶対に休みにする
  if staff.name in request_off and day in request_off[staff.name]:
        return False
  # 最大5連勤まで
  if streak(staff,day) >= 5:
        return False
  return True

 # スコア（小さいほど優先）
def score(staff):
    return staff.shift.count("出勤")

# 週2休チェック
def weekly_rest_ok(staff,day):
    week_start = (day // 7) * 7
    week_end = week_start + 7
    rest = staff.shift[week_start:week_end].count("公休")
    return rest >= 2

# メイン処理作成
def generate_shift(days=30,main_staff=6):
  _init(staffs,days)

  print("========= シフト作成中 ==========")

  for day in range(days):

  # 日曜は固定休
        if weekdays[day % 7] == "日":
            continue

        while sum(s.shift[day] == "出勤" for s in staffs) < main_staff:

            need_care = not any(
                s.shift[day] == "出勤" and s.license == "介護福祉士"
                for s in staffs
            )

            # 公平性：出勤が少ない順
            sorted_staff = sorted(staffs, key=score)

            filled = False

            for s in sorted_staff:

                if not can_work(s, day):
                    continue

                # 週2休保証チェック（無理ならスキップ）
                if not weekly_rest_ok(s, day):
                    continue

                # 介護福祉士が優先
                if need_care and s.license == "介護福祉士":
                    s.shift[day] = "出勤"
                    filled = True
                    print(f"{day+1}日 → {s.name}（介護福祉士）")
                    break

                elif not need_care:
                    s.shift[day] = "出勤"
                    filled = True
                    print(f"{day+1}日 → {s.name}")
                    break

            if not filled:
                print(f"{day+1}日 → ⚠ 補填不可、別拠点から支援要請")
                break




def print_calendar(staffs, days):
    print("\n=== シフト表 ===")

    # ヘッダー（日付）
    header = "名前  "
    for d in range(days):
        header += f"{d+1:>3}"
    print(header)

    # 曜日行
    week_row = "      "
    for d in range(days):
        week_row += f"{weekdays[d % 7]:>3}"
    print(week_row)

    # 各職員のシフト
    for s in staffs:
        row = f"{s.name:<4} "
        for d in range(days):
            if s.shift[d] == "出勤":
                mark = "○"
            elif s.shift[d] == "公休":
                mark = "-"
            else:
                mark = " "
            row += f"{mark:>3}"
        print(row)

generate_shift()
print_calendar(staffs, 30)


========= シフト作成中 ==========
1日 → 田中（介護福祉士）
1日 → 山田
1日 → 佐藤
1日 → 山本
1日 → 如月
1日 → 酒井
2日 → 田中（介護福祉士）
2日 → 中川
2日 → 田栗
2日 → 藤原
2日 → 青木
2日 → 山田
3日 → 田中（介護福祉士）
3日 → 佐藤
3日 → 山本
3日 → 如月
3日 → 酒井
3日 → 中川
4日 → 山田（介護福祉士）
4日 → 田栗
4日 → 藤原
4日 → 青木
4日 → 佐藤
4日 → 山本
5日 → 田中（介護福祉士）
5日 → 如月
5日 → 酒井
5日 → 中川
5日 → 田栗
5日 → 藤原
6日 → 山田（介護福祉士）
6日 → 青木
6日 → 佐藤
6日 → 山本
6日 → 如月
6日 → 酒井
8日 → 田中（介護福祉士）
8日 → 中川
8日 → 田栗
8日 → 藤原
8日 → 青木
8日 → 山田
9日 → 田中（介護福祉士）
9日 → 佐藤
9日 → 山本
9日 → 如月
9日 → 中川
9日 → 田栗
10日 → 山田（介護福祉士）
10日 → 酒井
10日 → 藤原
10日 → 青木
10日 → 佐藤
10日 → 山本
11日 → 田中（介護福祉士）
11日 → 如月
11日 → 酒井
11日 → 中川
11日 → 田栗
11日 → 藤原
12日 → 山田（介護福祉士）
12日 → 青木
12日 → 佐藤
12日 → 山本
12日 → 如月
12日 → 酒井
13日 → 田中（介護福祉士）
13日 → 中川
13日 → 田栗
13日 → 藤原
13日 → 青木
13日 → 山田
15日 → 田中（介護福祉士）
15日 → 佐藤
15日 → 山本
15日 → 如月
15日 → 酒井
15日 → 中川
16日 → 山田（介護福祉士）
16日 → 田栗
16日 → 藤原
16日 → 青木
16日 → 佐藤
16日 → 山本
17日 → 田中（介護福祉士）
17日 → 如月
17日 → 酒井
17日 → 中川
17日 → 田栗
17日 → 藤原
18日 → 山田（介護福祉士）
18日 → 青木
18日 → 佐藤
18日 → 山本
18日 → 如月
18日 → 酒井
19日 → 田中（介護福祉士）
19日 → 中川
19日 → 田栗
19日 → 藤原
1